# GVC-Local: Evaluate Fine-Tuned LoRA Model

This notebook evaluates the LoRA-fine-tuned Llama 3.1 8B model on held-out NYT Connections puzzles.

**Setup:** Runtime > Change runtime type > **T4 GPU**

- Base model: `unsloth/Llama-3.1-8B-Instruct` (ungated mirror of Meta's weights)
- LoRA adapter: `jacksonlukas/gvc-connections-lora`
- Quantization: 4-bit (NF4) via bitsandbytes
- Eval: basic solver loop on 50 held-out puzzles

In [ ]:
# Step 1: Install dependencies
!pip install -q torch transformers peft accelerate bitsandbytes requests

In [ ]:
# Step 2: Load base model + LoRA adapter
# Using unsloth's ungated mirror — identical weights to meta-llama/Llama-3.1-8B-Instruct
# but doesn't require Meta license approval.
import json
import random
import re
import time

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

random.seed(42)

BASE_MODEL = "unsloth/Llama-3.1-8B-Instruct"
ADAPTER_REPO = "jacksonlukas/gvc-connections-lora"

print(f"Loading base model: {BASE_MODEL}")
print(f"Loading adapter: {ADAPTER_REPO}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_REPO)
model.eval()
print("Model loaded successfully!")

In [ ]:
# Step 3: Define inference + parsing (matches training format from data_prep.py)

SYSTEM_PROMPT = """You are an expert at the NYT Connections puzzle. Given a list of 16 words \
(or fewer if some groups have been solved), identify groups of 4 related \
words and their category names.

You MUST format your response EXACTLY as follows:

<UNDERSTANDING_OF_BOARD>
Group1: word1, word2, word3, word4
Group2: word5, word6, word7, word8
Group3: word9, word10, word11, word12
Group4: word13, word14, word15, word16
<END_UNDERSTANDING_OF_BOARD>

<GUESS_FOR_THIS_ROUND>
Group: word_a, word_b, word_c, word_d
Category: category_name
<END_GUESS_FOR_THIS_ROUND>

Pick the group you are MOST confident about as your guess."""


def generate_response(user_prompt, max_new_tokens=512):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def parse_guess(reply):
    """Parse the model's structured reply into (group, category) or None."""
    guess_match = re.search(
        r"<GUESS_FOR_THIS_ROUND>(.*?)<END_GUESS_FOR_THIS_ROUND>",
        reply, re.DOTALL,
    )
    if not guess_match:
        return None
    guess_text = guess_match.group(1).strip()
    group_m = re.search(r"Group:\s*(.+)", guess_text, re.IGNORECASE)
    cat_m = re.search(r"Category:\s*(.+)", guess_text, re.IGNORECASE)
    if not group_m or not cat_m:
        return None
    group = [
        w.strip().upper().replace(",", "")
        for w in re.split(r",\s*", group_m.group(1))
        if w.strip()
    ]
    category = cat_m.group(1).strip()
    if len(group) != 4:
        return None
    return group, category


# Quick sanity check
print("Running sanity check...")
test_reply = generate_response(
    "Words: BASS, DRUM, GUITAR, PIANO, RED, BLUE, GREEN, YELLOW, "
    "APPLE, BANANA, CHERRY, GRAPE, DOG, CAT, BIRD, FISH"
)
print("=== Model output ===")
print(test_reply)
print("\nParsed guess:", parse_guess(test_reply))

In [ ]:
# Step 4: Load puzzles from public dataset
import requests

url = (
    "https://raw.githubusercontent.com/Eyefyre/NYT-Connections-Answers/"
    "refs/heads/main/connections.json"
)
all_puzzles = requests.get(url, timeout=30).json()
print(f"Loaded {len(all_puzzles)} puzzles total")

# Training used puzzles 0-829 (80% of 1037). Test on 830+.
TEST_START = 830
MAX_PUZZLES = 50
test_puzzles = all_puzzles[TEST_START : TEST_START + MAX_PUZZLES]
print(f"Evaluating {len(test_puzzles)} held-out puzzles ({TEST_START} to {TEST_START + len(test_puzzles) - 1})")

In [ ]:
# Step 5: Run evaluation (multi-round solver loop)
MAX_STRIKES = 20  # max failed attempts per puzzle before giving up

results = []
total_solved = 0
t0 = time.time()

for i, puzzle in enumerate(test_puzzles):
    categories = puzzle["answers"]
    remaining_cats = list(categories)
    strikes = 0
    solved = 0
    failed_guesses = []

    while remaining_cats and strikes < MAX_STRIKES:
        # Build prompt with remaining words
        remaining_words = [w for cat in remaining_cats for w in cat["members"]]
        random.shuffle(remaining_words)
        user_prompt = f"Words: {', '.join(remaining_words)}"

        reply = generate_response(user_prompt)
        result = parse_guess(reply)

        if result is None:
            strikes += 1
            continue

        group, category = result
        guess_set = frozenset(group)
        matched = False

        for j, cat in enumerate(remaining_cats):
            cat_set = frozenset(m.upper() for m in cat["members"])
            if guess_set == cat_set:
                solved += 1
                remaining_cats.pop(j)
                matched = True
                break

        if not matched:
            strikes += 1
            sg = sorted(group)
            if sg in failed_guesses:
                continue
            failed_guesses.append(sg)

    is_solved = solved == 4
    if is_solved:
        total_solved += 1

    row = {
        "puzzle_id": TEST_START + i,
        "solved": is_solved,
        "categories_solved": solved,
        "strikes": strikes,
    }
    results.append(row)
    status = "SOLVED" if is_solved else "FAILED"
    elapsed_so_far = time.time() - t0
    avg_per_puzzle = elapsed_so_far / (i + 1)
    eta = avg_per_puzzle * (len(test_puzzles) - i - 1)
    print(
        f"Puzzle {TEST_START + i} ({i+1}/{len(test_puzzles)}): {status}  "
        f"({solved}/4, {strikes} strikes)  "
        f"[{elapsed_so_far:.0f}s elapsed, ~{eta:.0f}s remaining]"
    )

elapsed = time.time() - t0
rate = total_solved / len(results) * 100

print(f"\n{'='*60}")
print(f"  Model:      {ADAPTER_REPO}")
print(f"  Puzzles:    {len(results)} (held-out, indices {TEST_START}+)")
print(f"  Solved:     {total_solved}/{len(results)} ({rate:.1f}%)")
print(f"  Avg cats:   {sum(r['categories_solved'] for r in results)/len(results):.2f}/4")
print(f"  Wall time:  {elapsed:.1f}s ({elapsed/len(results):.1f}s/puzzle)")
print(f"{'='*60}")

In [ ]:
# Step 6: Save results + breakdown
with open("finetuned_lora_eval.jsonl", "w") as f:
    for row in results:
        f.write(json.dumps(row) + "\n")

cat_counts = [r["categories_solved"] for r in results]
print("Category-level breakdown:")
print(f"  4/4 (full solve): {sum(1 for c in cat_counts if c == 4)}")
print(f"  3/4:              {sum(1 for c in cat_counts if c == 3)}")
print(f"  2/4:              {sum(1 for c in cat_counts if c == 2)}")
print(f"  1/4:              {sum(1 for c in cat_counts if c == 1)}")
print(f"  0/4:              {sum(1 for c in cat_counts if c == 0)}")

print(f"\nResults saved to finetuned_lora_eval.jsonl")
print("Download this file and copy it to gvc-local/results/")

In [ ]:
# Step 7 (optional): Run on first 10 puzzles too for direct comparison with baselines
# The baselines (basic 20%, snap_gvc 67%) were run on puzzles 0-9.
# Uncomment below to get a direct apples-to-apples comparison.

# COMPARE_PUZZLES = all_puzzles[0:10]
# compare_results = []
# compare_solved = 0
# random.seed(42)
#
# for i, puzzle in enumerate(COMPARE_PUZZLES):
#     categories = puzzle["answers"]
#     remaining_cats = list(categories)
#     strikes = 0
#     solved = 0
#     failed_guesses = []
#     while remaining_cats and strikes < 20:
#         remaining_words = [w for cat in remaining_cats for w in cat["members"]]
#         random.shuffle(remaining_words)
#         reply = generate_response(f"Words: {', '.join(remaining_words)}")
#         result = parse_guess(reply)
#         if result is None:
#             strikes += 1
#             continue
#         group, category = result
#         guess_set = frozenset(group)
#         matched = False
#         for j, cat in enumerate(remaining_cats):
#             cat_set = frozenset(m.upper() for m in cat["members"])
#             if guess_set == cat_set:
#                 solved += 1
#                 remaining_cats.pop(j)
#                 matched = True
#                 break
#         if not matched:
#             strikes += 1
#             sg = sorted(group)
#             if sg in failed_guesses:
#                 continue
#             failed_guesses.append(sg)
#     is_solved = solved == 4
#     if is_solved:
#         compare_solved += 1
#     print(f"Puzzle {i}: {'SOLVED' if is_solved else 'FAILED'} ({solved}/4, {strikes} strikes)")
#
# print(f"\nDirect comparison (puzzles 0-9):")
# print(f"  Fine-tuned basic: {compare_solved}/10 ({compare_solved*10}%)")
# print(f"  Baseline basic:   2/10 (20%)")
# print(f"  Snap-GVC:         6/9  (67%)")